In [40]:
import json

from collections import defaultdict

In [41]:
with open("./sallm_nl_prompt_gpt_translated_prompt_gpt_nl_prompt_retranslated.jsonl") as f:
    data = [json.loads(line) for line in f.readlines()]
len(data)

100

In [42]:
new_data = []
skipped_translations_1 = defaultdict(int)
skipped_translations_2 = defaultdict(int)
for item in data:
    id = item["id"]
    technique = item["technique"]
    source = item["source"]
    english_prompt = item["prompt"]
    insecure_code = item["insecure_code"]
    nl_descriptions = item["prompt_nl_prompt"].split("|")

    # Fine the start and end line number where the nl_description is located in the english prompt
    start_line = []
    end_line = []
    for nl_description in nl_descriptions:
        nl_description = nl_description.strip()
        start_line.append(english_prompt.find(nl_description))
        end_line.append(start_line[-1] + len(nl_description))
   
    # print(id, start_line, end_line)
    for translation in item["translations"]:
        current_translation = item["translations"][translation][0]["translation"].split("|")
        if len(current_translation) != len(nl_descriptions):
            skipped_translations_1[translation] += 1
            continue


        if '2023' in item["translations"][translation][0]["translation"] or '2023' in item["translations"][translation][0]["back_translation"] or "English" in translation:
            skipped_translations_2[translation] += 1
            continue
        

        if len(current_translation) == 1:
            new_prompt = english_prompt[:start_line[0]] + current_translation[0] + english_prompt[end_line[0]:]
        else:

            new_prompt = english_prompt[:start_line[0]]
            for i in range(len(nl_descriptions)):
                if i == len(nl_descriptions) - 1:
                    new_prompt += current_translation[i] + english_prompt[end_line[i]:]
                else:
                    new_prompt += current_translation[i] + english_prompt[end_line[i]:start_line[i+1]]
        new_data.append({
                "id": id,
                "technique": technique,
                "source": source,
                "prompt": new_prompt,
                "insecure_code": insecure_code,
                "language": translation,
                "translated_prompt": new_prompt
            })
        

len(new_data)

1936

In [43]:
# Print the skipped translations and their counts in a table format
print("Skipped Translations:")
total_skipped = 0
for translation, count in skipped_translations_1.items():
    total_skipped += count
    print(f"{translation}: {count}")

total_skipped = total_skipped - skipped_translations_1["English"]

print("\nSkipped Translations:", len(skipped_translations_1.keys())-1, total_skipped, total_skipped/(len(skipped_translations_1.keys())-1))
print("-" * 30)
total_skipped = 0
for translation, count in skipped_translations_2.items():
    total_skipped += count
    print(f"{translation}: {count}")

total_skipped = total_skipped - skipped_translations_2["English"]

print("\nSkipped Translations:", len(skipped_translations_2.keys())-1, total_skipped, total_skipped/(len(skipped_translations_2.keys())-1))
print("-" * 30)

Skipped Translations:
Acehnese: 7
Hebrew: 5
Vietnamese: 5
Indonesian: 2
Malayalam: 4
Tagalog: 3
English: 2
Dutch: 4
German: 4
Afrikaans: 5
Portuguese: 4
Spanish: 4
French: 4
Italian: 3
Greek: 3
Western Persian: 5
Russian: 5
Bulgarian: 4
Chinese: 4
Turkish: 3
Estonian: 5
Finnish: 5
Hungarian: 2

Skipped Translations: 22 90 4.090909090909091
------------------------------
English: 98
Malayalam: 21
Dutch: 6
Afrikaans: 12
Portuguese: 13
Spanish: 4
French: 8
Greek: 6
Russian: 4
Bulgarian: 11
Chinese: 6
Estonian: 9
Finnish: 8
Acehnese: 22
Vietnamese: 7
Indonesian: 6
Tagalog: 7
German: 4
Italian: 2
Turkish: 3
Hungarian: 3
Western Persian: 5
Hebrew: 7

Skipped Translations: 22 174 7.909090909090909
------------------------------
